Second model, second generative family. Drax is discrete flow matching; Whisfusion is masked diffusion.

**GPU T4 x2**, Internet on, attach `prepare-data-for-word-reranker`. Run the cells one at a time — the install is the risky part.

In [1]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [2]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)
K.gpu_info()
env = K.prepare(COMMIT)

Cloning into '/kaggle/working/candidate_reranker'...
From https://github.com/Splestule/candidate_reranker
 * branch            main       -> FETCH_HEAD


HEAD is now at da83949 Merge remote-tracking branch 'origin/main'
torch 2.10.0+cu128  cuda=True
Tesla T4  compute capability 7.5
Turing/Pascal: no bf16, no FlashAttention 2 -- handled by wf_compat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 882.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.1 MB/s eta 0:00:00


Cloning into '/kaggle/working/Whisfusion'...


commit      da83949
base model  /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ckpt/mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors
adapter     /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ckpt/whisfusion_stage2_decoder.pt
librispeech /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/data/LibriSpeech  ['dev-clean', 'test-clean', 'test-other']
hf cache    /kaggle/working/hf  prepared
ood         /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ood  ['irish_english_male', 'midlands_english_female', 'northern_english_female']


Drax pins `torch==2.7.0`. Kaggle ships a newer one and downgrading it breaks CUDA, so this installs **without dependencies** on purpose.

In [3]:
import shutil

old = Path("/kaggle/working/drax")
if old.exists():
    shutil.rmtree(old)

DRAX = Path("/kaggle/working/drax_repo")
if not DRAX.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/aiola-lab/drax.git", str(DRAX)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(DRAX), "--no-deps"],
               check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "omegaconf>=2.3.0", "einops>=0.7.0", "flow-matching>=1.0.6",
                "safetensors>=0.4.2", "huggingface_hub>=0.24.0"], check=True)
print("hotovo")

Cloning into '/kaggle/working/drax_repo'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 1.6 MB/s eta 0:00:00
hotovo


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
drax-asr 0.1.0 requires transformers==4.52.3, but you have transformers 5.0.0 which is incompatible.


In [4]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.3"], check=True)
print("OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.2 MB/s eta 0:00:00
OK


Probe the import. Whatever it reports missing goes into the next cell — fail fast, one dependency at a time.

In [5]:
import sys, subprocess
from pathlib import Path

DRAX = Path("/kaggle/working/drax_repo")

# diagnostika, ne hadani
print("slozka existuje:", DRAX.exists())
print("balik uvnitr:   ", (DRAX / "drax" / "__init__.py").exists())

if not DRAX.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/aiola-lab/drax.git", str(DRAX)], check=True)

if str(DRAX) not in sys.path:
    sys.path.insert(0, str(DRAX))

import drax
print("\ndrax from:", drax.__file__)
from drax import Transcriber
print("Transcriber OK")


slozka existuje: True
balik uvnitr:    True

drax from: /kaggle/working/drax_repo/drax/__init__.py
Transcriber OK


One utterance, k = 4, to prove it runs on a T4 at all and that the candidates actually differ. If this is out of memory, drop `k`; if it is slow, drop `sampling_steps`.

In [6]:
import torch
from drax import Transcriber

asr = Transcriber(model_path="aiola/drax-v1")

import data as dataio, itertools
u = next(itertools.islice(dataio.iter_utterances("librispeech", env.librispeech / "test-clean"), 1))
print("REF", u.text, "\n")

res = asr.transcribe([u.audio_path] * 4, language=["en"] * 4,
                     sampling_steps=8, temperature=0.6)
for i, r in enumerate(res):
    print(i, r.transcript)

print(f"\nunikatnich: {len({r.transcript for r in res})} ze 4")
print(f"spicka pameti: {torch.cuda.max_memory_allocated() / 2**20:.0f} MB")

config.json: 0.00B [00:00, ?B/s]

encoder.safetensors:   0%|          | 0.00/2.55G [00:00<?, ?B/s]

decoder.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

REF HE HOPED THERE WOULD BE STEW FOR DINNER TURNIPS AND CARROTS AND BRUISED POTATOES AND FAT MUTTON PIECES TO BE LADLED OUT IN THICK PEPPERED FLOUR FATTENED SAUCE 

0  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes, and fat mutton pieces to be ladled out in thick, peppered, flour fatted sauce
1  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces, be be ladled out, in thick peppered flour f feded sauce
2  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes, and fat mutton pieces to be ladled out in thick, peppered, flour fattened sauce,
3  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes, and fat mutton pieces to be ladled out in thick, peppered, flour fattened sauce,

unikatnich: 3 ze 4
spicka pameti: 7184 MB


Full smoke test: 40 utterances up to 15 s, four temperatures. The candidates only diverge if the temperature is high enough, and run 1 warned that random diversity can hurt — so this sweeps it rather than guessing.

In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

if "/kaggle/working/drax_repo" not in env.pythonpath:
    env.pythonpath += ":/kaggle/working/drax_repo"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

rc = K.run(env, "drax_smoke.py", ...)


rc = K.run(env, "drax_smoke.py",
           "--librispeech", env.librispeech / "test-clean",
           "--out_dir", env.results,
           "--n_utts", 40, "--k", 10,
           "--sampling_steps", 8,
           "--temperatures", "0.1,0.3,0.6,1.0")
assert rc == 0, f"drax_smoke exit code {rc}, see the output above"

usage: drax_smoke.py [-h] [--model_path MODEL_PATH] --librispeech LIBRISPEECH
                     --out_dir OUT_DIR [--n_utts N_UTTS] [--k K]
                     [--sampling_steps SAMPLING_STEPS]
                     [--temperatures TEMPERATURES] [--language LANGUAGE]
                     [--max_seconds MAX_SECONDS]
drax_smoke.py: error: the following arguments are required: --librispeech, --out_dir
40 utterances, 4.4 min of audio, k = 10, 8 sampling steps

  t=0.1  10/40  142 s
  t=0.1  20/40  294 s
  t=0.1  30/40  445 s
  t=0.1  40/40  596 s
  t=0.1 done in 596 s -> /kaggle/working/results/drax-t0.1-k10.jsonl

  t=0.3  10/40  152 s
  t=0.3  20/40  304 s
  t=0.3  30/40  456 s
  t=0.3  40/40  608 s
  t=0.3 done in 608 s -> /kaggle/working/results/drax-t0.3-k10.jsonl

  t=0.6  10/40  152 s
  t=0.6  20/40  305 s
  t=0.6  30/40  457 s
  t=0.6  40/40  608 s
  t=0.6 done in 608 s -> /kaggle/working/results/drax-t0.6-k10.jsonl

  t=1.0  10/40  152 s
  t=1.0  20/40  305 s
  t=1.0  30/40  45